In [1]:
import numpy as np
import os
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from scipy.linalg import fractional_matrix_power

from pyscf import gto, scf, lib
from ipie.utils.from_pyscf import gen_ipie_input_from_pyscf_chk
from ipie.qmc.afqmc import AFQMC
from ipie.estimators.estimator_base import EstimatorBase
from ipie.estimators.energy import local_energy_batch
from ipie.utils.mpi import MPIHandler

# Check for GPU (CuPy) support
try:
    import cupy as cp
    has_cupy = True
except ImportError:
    cp = None
    has_cupy = False

# =============================================================================
# 0. CONFIGURATION & ML LOADING
# =============================================================================
N_ATOMS = 10            
SYSTEM_NAME = f"H{N_ATOMS}"
WALKERS = 2048           
DEPLOY_DIR = "deployment_objects"
MODEL_PATH = os.path.join(DEPLOY_DIR, f"NN_{SYSTEM_NAME}_DeltaHF.keras")

@tf.keras.utils.register_keras_serializable()
def log_cosh_loss(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, tf.shape(y_pred)), dtype=y_pred.dtype)
    return tf.reduce_mean(tf.math.log(tf.math.cosh(y_pred - y_true)))

print(f">>> STARTING ML COMPARATOR FOR: {SYSTEM_NAME}")

if not os.path.exists(DEPLOY_DIR):
    raise FileNotFoundError(f"Deployment folder '{DEPLOY_DIR}' not found!")

print("    Loading ML Deployment Assets...")
ml_model = load_model(MODEL_PATH, custom_objects={"log_cosh_loss": log_cosh_loss})
X_scaler = joblib.load(os.path.join(DEPLOY_DIR, "X_scaler.save"))
y_scaler = joblib.load(os.path.join(DEPLOY_DIR, "y_scaler.save"))

P_hf_ref = np.load(os.path.join(DEPLOY_DIR, "P_hf_lowdin.npy"))

# =============================================================================
# 1. DYNAMIC HELPER FUNCTIONS
# =============================================================================
def get_dynamic_operators(mol):
    S = mol.intor('int1e_ovlp')
    h_core_ao = mol.intor('int1e_nuc') + mol.intor('int1e_kin')
    
    e, v = np.linalg.eigh(S)
    mask = e > 1e-15
    
    # S^(-1/2) and S^(+1/2)
    S_inv_sqrt = v[:, mask] @ np.diag(e[mask]**(-0.5)) @ v[:, mask].T
    S_sqrt = v[:, mask] @ np.diag(e[mask]**(0.5)) @ v[:, mask].T
    
    # Lowdin H_core
    h_core_lowdin = S_inv_sqrt.T @ h_core_ao @ S_inv_sqrt
    
    return S_inv_sqrt, S_sqrt, h_core_lowdin, S 

# =============================================================================
# 2. REAL-TIME ML COMPARATOR ESTIMATOR (CORRECTED)
# =============================================================================
class MLComparator(EstimatorBase):
    def __init__(self, ham, trial, h_core_dyn, E_nuc_dyn, P_hf_ref, E_hf_ref, C_a, S_ao, S_sqrt, verbose=False):
        super().__init__()
        self._shape = (1,) 
        self._scalar_estimator = True 
        
        # NOTE: We now process ALL walkers to match ipie output exactly
        self.counter = 0
        self.verbose = verbose
        
        # Operators
        self.h_core_dyn = h_core_dyn
        self.E_nuc_dyn = E_nuc_dyn
        self.S_ao = S_ao
        self.S_sqrt = S_sqrt
        self.C_a = C_a

        # References
        self.P_hf_ref = P_hf_ref
        self.E_hf_ref = E_hf_ref

        # Trial Wfn info
        self.nalpha = trial.nalpha
        self.nbeta  = trial.nbeta
        self.Psi_T_a = trial.psi[:, :trial.nalpha].astype(np.complex128)
        self.Psi_T_b = trial.psi[:, trial.nalpha:].astype(np.complex128)

        # GPU Setup
        self.use_gpu = has_cupy 
        if self.use_gpu:
            self.cp = cp
            self.Psi_T_a_gpu = cp.asarray(self.Psi_T_a)
            self.Psi_T_b_gpu = cp.asarray(self.Psi_T_b)
            self.C_a_gpu = cp.asarray(self.C_a)
        else:
            self.cp = None

        self.all_ipie_energies = []
        self.all_ml_energies = []
        
        # Rank logic
        self.rank = 0 
        try:
            from mpi4py import MPI
            self.rank = MPI.COMM_WORLD.Get_rank()
        except: pass

        @tf.function(reduce_retracing=True)
        def fast_predict(inputs):
            return ml_model(inputs, training=False)
        self.fast_predict = fast_predict

    def compute_estimator(self, system, hamiltonian, trial, walkers):
        self.counter += 1
        
        # =====================================================================
        # APPLES-TO-APPLES: USE ALL WALKERS
        # =====================================================================
        n_samp = walkers.nwalkers  # Explicitly use ALL walkers
        
        # 1. Gather Walker Data & Weights
        if hasattr(walkers, 'phia'):
            phi_a, phi_b = walkers.phia[:n_samp], walkers.phib[:n_samp]
        else:
            phi_a = walkers.phi[:n_samp, :, :self.nalpha]
            phi_b = walkers.phi[:n_samp, :, self.nalpha:]

        # [CRITICAL] Extract Complex Weights
        weights = walkers.weight[:n_samp]
        if hasattr(weights, 'get'): weights = weights.get()

        # 2. Exact 1-RDM Construction
        on_gpu = self.use_gpu and (hasattr(walkers.weight, 'device') or isinstance(walkers.weight, self.cp.ndarray))
        
        if on_gpu:
            if not isinstance(phi_a, self.cp.ndarray): phi_a = self.cp.array(phi_a)
            if not isinstance(phi_b, self.cp.ndarray): phi_b = self.cp.array(phi_b)
            xp = self.cp
            Psi_a, Psi_b = self.Psi_T_a_gpu, self.Psi_T_b_gpu
            C_sim = self.C_a_gpu 
        else:
            if hasattr(phi_a, 'get'): phi_a = phi_a.get()
            if hasattr(phi_b, 'get'): phi_b = phi_b.get()
            xp = np
            Psi_a, Psi_b = self.Psi_T_a, self.Psi_T_b
            C_sim = self.C_a 

        O_a = xp.einsum('ui, wuj -> wij', Psi_a.conj(), phi_a)
        O_b = xp.einsum('ui, wuj -> wij', Psi_b.conj(), phi_b)
        invO_a = xp.linalg.inv(O_a)
        invO_b = xp.linalg.inv(O_b)
        right_a = xp.einsum('wij, ju -> wiu', invO_a, Psi_a.conj().T)
        right_b = xp.einsum('wij, ju -> wiu', invO_b, Psi_b.conj().T)
        G_mo_a = xp.einsum('wvi, wiu -> wvu', phi_a, right_a)
        G_mo_b = xp.einsum('wvi, wiu -> wvu', phi_b, right_b)
        tmp_a = xp.einsum("wvu, ku -> wvk", G_mo_a, C_sim.conj())
        tmp_b = xp.einsum("wvu, ku -> wvk", G_mo_b, C_sim.conj())
        P_ao_a = xp.einsum("qv, wvk -> wqk", C_sim, tmp_a)
        P_ao_b = xp.einsum("qv, wvk -> wqk", C_sim, tmp_b)
        
        if on_gpu:
            P_ao_a = self.cp.asnumpy(P_ao_a)
            P_ao_b = self.cp.asnumpy(P_ao_b)
        
        P_total_ao = P_ao_a + P_ao_b

        # 3. IPIE Energy Calculation (Full Batch)
        local_E_full = local_energy_batch(system, hamiltonian, walkers, trial)[:n_samp]
        if hasattr(local_E_full, 'get'): local_E_full = local_E_full.get()
        E_total_ipie_local = local_E_full[:, 0] # Complex

        # 4. ML Inference (Full Batch)
        # Note: 2048 samples is small for a modern CNN, so this is fast.
        P_total_lowdin = np.einsum('ai, wib, bj -> waj', self.S_sqrt, P_total_ao, self.S_sqrt)
        delta_P = P_total_lowdin - self.P_hf_ref
        
        X_real, X_imag = np.real(delta_P), np.imag(delta_P)
        X_input = np.stack([X_real, X_imag], axis=-1).astype(np.float32)
        X_flat = X_input.reshape(n_samp, -1)
        X_scaled = X_scaler.transform(X_flat).reshape(n_samp, N_ATOMS, N_ATOMS, 2)

        preds_scaled = self.fast_predict(X_scaled).numpy()
        E_corr_delta = y_scaler.inverse_transform(preds_scaled).flatten()

        E_1B_delta = np.einsum('ij, bji -> b', self.h_core_dyn, delta_P).real
        E_total_ml_local = self.E_hf_ref + E_1B_delta + E_corr_delta

        # 5. [CORRECTED] Weighted Average
        # This matches ipie's internal estimator logic: Real( Sum(W*E) / Sum(W) )
        denom = np.sum(weights)
        
        # Calculate Weighted Means
        avg_ipie = np.real(np.sum(weights * E_total_ipie_local) / denom)
        avg_ml   = np.real(np.sum(weights * E_total_ml_local) / denom)

        self.all_ipie_energies.append(avg_ipie)
        self.all_ml_energies.append(avg_ml)

        if self.counter % 5 == 0 and self.rank == 0:
            batch_mae = np.abs(avg_ipie - avg_ml) * 1000
            print(f"    Block {self.counter:03d} | ipie: {avg_ipie:.4f} Ha | ML: {avg_ml:.4f} Ha | MAE: {batch_mae:.2f} mHa")

    @property
    def names(self): return ["MLComparator"]
    @property
    def shape(self): return self._shape
    @property
    def data(self):  return np.zeros(self._shape, dtype=np.complex128)

# =============================================================================
# 3. SETUP & RUN
# =============================================================================
comm = MPIHandler()
rank = comm.rank

chk_file = f"scf_h{N_ATOMS}.chk"
ham_file = f"ham_h{N_ATOMS}.h5"
wfn_file = f"wfn_h{N_ATOMS}.h5"

if rank == 0:
    mol = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
    mf = scf.UHF(mol)
    mf.chkfile = chk_file
    
    # 1. Ensure PySCF has run
    if not os.path.exists(chk_file) or not os.path.exists(wfn_file):
        print(f"[Main] Running PySCF and generating integrals...")
        mf.kernel()
        gen_ipie_input_from_pyscf_chk(chk_file, hamil_file=ham_file, wfn_file=wfn_file, verbose=0, chol_cut=1e-5)
    
    # 2. [ROBUST] Load Energy from Checkpoint
    print(f"[Main] Reading HF Energy from: {chk_file}")
    CURRENT_HF_ENERGY = lib.chkfile.load(chk_file, 'scf/e_tot')
    
    if CURRENT_HF_ENERGY is None:
        print("[Main] Warning: 'e_tot' missing in chkfile. Re-running kernel...")
        CURRENT_HF_ENERGY = mf.kernel()

    print(f"[Main] Current PySCF HF Energy: {CURRENT_HF_ENERGY:.6f} Ha")

else:
    CURRENT_HF_ENERGY = None

comm.comm.Barrier()
CURRENT_HF_ENERGY = comm.comm.bcast(CURRENT_HF_ENERGY, root=0)

# --- LOAD DYNAMIC COEFFICIENTS & OPERATORS ---
if rank == 0:
    print(f"\n[Main] Loading Dynamic MO Coefficients from: {chk_file}")
    mo_coeff = lib.chkfile.load(chk_file, 'scf/mo_coeff')
    if np.ndim(mo_coeff) == 3: C_a, C_b = mo_coeff[0], mo_coeff[1]
    else: C_a = C_b = mo_coeff

    mol_rt = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
    S_inv_sqrt, S_sqrt, h_core_dyn, S_ao = get_dynamic_operators(mol_rt)
    E_nuc_dyn = mol_rt.energy_nuc()

# Re-compute on all ranks
mol_rt = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
S_inv_sqrt, S_sqrt, h_core_dyn, S_ao = get_dynamic_operators(mol_rt)
E_nuc_dyn = mol_rt.energy_nuc()

mo_coeff = lib.chkfile.load(chk_file, 'scf/mo_coeff')
if np.ndim(mo_coeff) == 3: C_a, C_b = mo_coeff[0], mo_coeff[1]
else: C_a = C_b = mo_coeff

TEST_SEED = 999 

if rank == 0:
    print(f"\n" + "="*50)
    print(f">>> Starting Evaluation Run (Seed: {TEST_SEED})")
    print("="*50)

afqmc = AFQMC.build_from_hdf5(
    num_elec=(N_ATOMS//2, N_ATOMS//2),
    ham_file=ham_file,
    wfn_file=wfn_file,
    num_blocks=50,              
    num_steps_per_block=1,
    num_walkers=WALKERS,
    seed=TEST_SEED,             
    verbose=0
)

afqmc.mpi_handler = comm
if comm.size > 1:
    local_walkers = WALKERS // comm.size
    if rank < (WALKERS % comm.size): local_walkers += 1
    afqmc.nwalkers = local_walkers

# [CORRECTION] Removed 'num_samples=10'. Now processes ALL walkers.
comparator = MLComparator(
    afqmc.hamiltonian, 
    afqmc.trial, 
    h_core_dyn=h_core_dyn,
    E_nuc_dyn=E_nuc_dyn,
    P_hf_ref=P_hf_ref,
    E_hf_ref=CURRENT_HF_ENERGY, 
    C_a=C_a,
    S_ao=S_ao,
    S_sqrt=S_sqrt
)

afqmc.run(additional_estimators={"ML_Test": comparator})

if rank == 0:
    print("\n" + "="*50)
    print(">>> FINAL RUN SUMMARY")
    print("="*50)
    
    # Skip equilibration (first 10)
    final_ipie = np.mean(comparator.all_ipie_energies[10:])
    final_ml = np.mean(comparator.all_ml_energies[10:])
    final_mae = np.abs(final_ipie - final_ml) * 1000

    print(f"    Avg IPIE Total Energy: {final_ipie:.6f} Ha")
    print(f"    Avg ML Total Energy:   {final_ml:.6f} Ha")
    print(f"    Final Deployment MAE:  {final_mae:.4f} mHa")

2026-02-27 12:46:58.487239: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


>>> STARTING ML COMPARATOR FOR: H10
    Loading ML Deployment Assets...


I0000 00:00:1772214426.625293 1648093 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1036 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:23:00.0, compute capability: 8.6


[Main] Reading HF Energy from: scf_h10.chk
[Main] Current PySCF HF Energy: -5.096510 Ha

[Main] Loading Dynamic MO Coefficients from: scf_h10.chk

>>> Starting Evaluation Run (Seed: 999)
# random seed is 999
            Block                   Weight            WeightFactor            HybridEnergy                  ENumer                  EDenom                  ETotal                  E1Body                  E2Body
                0   2.0480000000000000e+03  2.0480000000000000e+03  0.0000000000000000e+00 -1.0437649726627384e+04  2.0480000000000000e+03 -5.0965086555797772e+00 -1.7754760831862487e+01  1.2658252176282712e+01
                1   2.0616780949217637e+03  2.0480000000000000e+03 -2.6630942934985731e+00 -1.0509588414829544e+04  2.0616780949217637e+03 -5.0975894057934203e+00 -1.7754758987919363e+01  1.2657169582125944e+01
                2   2.0617445384773637e+03  2.0480000000000000e+03 -2.6759482301032134e+00 -1.0512091286110022e+04  2.0617445384773637e+03 -5.0986390844878366e

In [2]:
from ipie.analysis.autocorr import reblock_by_autocorr
from ipie.analysis.extraction import extract_observable
qmc_data = extract_observable(afqmc.estimators.filename, "energy")
y = qmc_data["ETotal"][1:]
df = reblock_by_autocorr(y, verbose=1)
print(df)

# Reblock based on autocorrelation time
nsamples, tac = 3, 0.0
nsamples, tac = 6, 0.715841893421713
nsamples, tac = 12, 1.1188053511055145
nsamples, tac = 25, 3.4095802244124753
nsamples, tac = 50, 7.130544699165011
   ETotal_ac  ETotal_error_ac  ETotal_nsamp_ac  ac
0  -5.118546         0.004539                6   8
